# MCP client

This notebook connects a chatbot to the research server. The server owns the tool list and the tool calls. The chatbot sends each user message and waits until the user types `quit`.

The server file is [`research_server.py`](./research_server.py). The in-process tool loop is in [direct tool calling](./mcp.ipynb). The client file is [`mcp_client.py`](./mcp_client.py).

## Related notes

- [Direct tool calling](./mcp.ipynb)
- [MCP server](./mcp-server.ipynb)
- [Reference servers](./reference-servers.ipynb)
- [Prompts and resources](./prompts-and-resources.ipynb)
- [Subprocess](./subprocess.ipynb)


## Session shape

The client keeps three things.

- A `ClientSession` connected to one server.
- An Anthropic client.
- `available_tools`, the schemas returned by that server.

`connect_to_server_and_run` starts `research_server.py` as a child process. It stores the tool schemas and passes them to the model.

When the model requests a tool, the session calls `call_tool`. The tool result goes back to the model.

The model name is `claude-sonnet-4-6`.


## Client session

This is the stdio pattern the class uses.

```python
server_params = StdioServerParameters(
    command="uv",
    args=["run", "--with", "mcp>=1.2,<2", "--with", "arxiv", "research_server.py"],
)

async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools = await session.list_tools()
        result = await session.call_tool("tool-name", arguments={"arg1": "value"})
```

`read` is the stream of server messages. `write` is the stream of client messages.


## Client code

The cell matches [`mcp_client.py`](./mcp_client.py), except the file also runs `main` when you execute it as a script. Do not run this cell if you only want to read it. Importing it loads the API key from `.env`.


In [ ]:
"""Single-server MCP client for research_server.py."""

import asyncio

from anthropic import Anthropic
from dotenv import load_dotenv
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

load_dotenv()


class MCP_ChatBot:
    def __init__(self):
        self.session: ClientSession = None
        self.anthropic = Anthropic()
        self.available_tools: list[dict] = []

    async def process_query(self, query):
        messages = [{"role": "user", "content": query}]
        response = self.anthropic.messages.create(
            max_tokens=2024,
            model="claude-sonnet-4-6",
            tools=self.available_tools,
            messages=messages,
        )
        process_query = True
        while process_query:
            assistant_content = []
            for content in response.content:
                if content.type == "text":
                    print(content.text)
                    assistant_content.append(content)
                    if len(response.content) == 1:
                        process_query = False
                elif content.type == "tool_use":
                    assistant_content.append(content)
                    messages.append({"role": "assistant", "content": assistant_content})
                    print(f"Calling tool {content.name} with args {content.input}")
                    result = await self.session.call_tool(
                        content.name, arguments=content.input
                    )
                    messages.append(
                        {
                            "role": "user",
                            "content": [
                                {
                                    "type": "tool_result",
                                    "tool_use_id": content.id,
                                    "content": result.content,
                                }
                            ],
                        }
                    )
                    response = self.anthropic.messages.create(
                        max_tokens=2024,
                        model="claude-sonnet-4-6",
                        tools=self.available_tools,
                        messages=messages,
                    )
                    if (
                        len(response.content) == 1
                        and response.content[0].type == "text"
                    ):
                        print(response.content[0].text)
                        process_query = False

    async def chat_loop(self):
        """Run an interactive chat loop."""
        print("\nMCP Chatbot Started!")
        print("Type your queries or 'quit' to exit.")
        while True:
            try:
                query = input("\nQuery: ").strip()
                if query.lower() == "quit":
                    break
                await self.process_query(query)
                print("\n")
            except Exception as error:
                print(f"\nError: {error}")

    async def connect_to_server_and_run(self):
        server_params = StdioServerParameters(
            command="uv",
            args=[
                "run",
                "--with",
                "mcp>=1.2,<2",
                "--with",
                "arxiv",
                "research_server.py",
            ],
        )
        async with stdio_client(server_params) as (read, write):
            async with ClientSession(read, write) as session:
                self.session = session
                await session.initialize()
                response = await session.list_tools()
                print(
                    "\nConnected to server with tools:",
                    [tool.name for tool in response.tools],
                )
                self.available_tools = [
                    {
                        "name": tool.name,
                        "description": tool.description,
                        "input_schema": tool.inputSchema,
                    }
                    for tool in response.tools
                ]
                await self.chat_loop()


async def main():
    chatbot = MCP_ChatBot()
    await chatbot.connect_to_server_and_run()


## Run it

From `MCP/`, with `ANTHROPIC_API_KEY` in an untracked `.env` file:

```bash
uv run --with "mcp>=1.2,<2" --with arxiv --with anthropic --with python-dotenv mcp_client.py
```

Type `quit` to exit. Paper files land in [`papers/`](./papers).

The next notebook replaces the one hardcoded server with [`server_config.json`](./server_config.json).

Model text can differ from run to run.


## Further reading

- [Client quickstart](https://modelcontextprotocol.io/quickstart/client)
- [Python SDK chatbot](https://github.com/modelcontextprotocol/python-sdk/blob/main/examples/clients/simple-chatbot/mcp_simple_chatbot/main.py)
